In [ ]:
import os, sys, pathlib
# Work from the repo root: find the folder containing `src/`, put it on the import path,
# and chdir into it so imports AND relative paths (configs/, data/, outputs/) resolve the
# same as running a script from the project root.
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Drop any stale/namespace `src`/`scripts` cached by an earlier failed import.
for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

print('working dir:', ROOT)

In [ ]:
from src.predict import run_all
from src.config import resolve_config_path, Config

cfg = Config.from_yaml(resolve_config_path("baseline_pilot"))

run_all(cfg)

In [ ]:
from pathlib import Path
import numpy as np
from src.data import MODULATION_CLASSES

RUNS = Path("runs")
files = sorted(RUNS.glob("*/*/predictions.npz"))
print(len(files), "cells")
for f in files:
    print(" ", f.parent.relative_to(RUNS))

In [ ]:
d = np.load(files[0])
pred, true, snr = d["pred"], d["true"], d["snr"]
print(pred.shape, true.shape, snr.shape)
print("класи в true:", np.unique(true).size, "з", len(MODULATION_CLASSES))
print("класи в pred:", np.unique(pred).size)
print("SNR:", np.unique(snr))

In [ ]:
acc = (pred == true).mean()
mask = snr >= 0
print(f"усі кадри:   {acc:.4f}")
print(f"SNR >= 0 dB: {(pred[mask] == true[mask]).mean():.4f}")

In [ ]:
for s in np.unique(snr):
    m = snr == s
    print(f"{s:>4} dB  n={m.sum():>6}  acc={(pred[m] == true[m]).mean():.3f}")

In [ ]:
for c in range(len(MODULATION_CLASSES)):
    m = mask & (true == c)
    if m.sum():
        print(f"{MODULATION_CLASSES[c]:>10}  n={m.sum():>5}  recall={(pred[m] == c).mean():.3f}")

In [ ]:
for f in files:
    z = np.load(f)
    m = z["snr"] >= 0
    print(f.parent.name, f"{(z['pred'][m] == z['true'][m]).mean():.4f}")